In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import csv
from functools import partial

# Prerequisites

We need to build the GHC binaries, run them and log the results. You must have the [Siege repository](https://github.com/haflang/heron) cloned locally, but not necessarily built yet.

The nix package manager can be used to setup a suitable environment, run the benchmarks, and write log files to `./ghc/{par,seq}/*.log` with:

```
> nix-shell
> nix shell <siege_dir>/heron#heron-clash <siege_dir>/heron#flite
> make
> for bench in `find ghc/par/* -executable`; do ./ghc_speedups.sh    -f $bench > $bench.log; done
> for bench in `find ghc/seq/* -executable`; do ./ghc_speedups.sh -b -f $bench > $bench.log; done
```

## Load full GHC results

Load all GHC csv files and insert derived columns.

In [ ]:
def selectMedians(df):
    meds = []
    for n in df['cores'].unique():
        runs = df[df['cores']==n]
        med = runs[runs['total']==runs['total'].median()]
        # NB this median filtering might not work for an even number of iterations...
        # I think pandas will report the mean between the two medians.
        meds.append(med)
    return pd.concat(meds)

def parseBench(basedir,b):
    pars = selectMedians(pd.read_csv(f'{basedir}/ghc/par/{b}.log'))
    baseline = selectMedians(pd.read_csv(f'{basedir}/ghc/seq/{b}.log'))['total'].iloc[0]
    pars['baseline'] = baseline
    pars['speedup'] = baseline/pars['total']
    pars['ideal'] = pars['speedup']/pars['cores']
    pars['bench'] = b
    return pars

def exportBenchCsvs(df, basename):
    benches = np.unique(df['bench'])
    for b in benches:
        df[df['bench']==b].to_csv(f'{basename}_{b}.csv', index=False, na_rep='nan')

from pathlib import Path

def importBenchCsvs(basename):
    # Find all matching CSV files
    csv_files = sorted(Path('.').glob(f'{basename}_*.csv'))
    
    # Read and concatenate all files
    dfs = [pd.read_csv(file) for file in csv_files]
    combined_df = pd.concat(dfs, ignore_index=True)
    
    return combined_df

In [ ]:
benches=!for f in ./ghc/par/*.log; do echo $(basename -s '.log' $f); done
df = pd.concat([parseBench('.',b) for b in benches])

# Plot and export

Display graphs for speedup (vs sequential version), % of ideal speedup obtained, and the productivity for each benchmark. We also export CSV files for use in LaTeX later.

In [ ]:
display(px.line(df, x='cores', y='speedup', color='bench'))
display(px.line(df, x='cores', y='ideal', color='bench'))
display(px.line(df, x='cores', y='productivity', color='bench'))
display(px.line(df, x='cores', y='gc', color='bench'))

In [ ]:
!mkdir csv_ghc
exportBenchCsvs(df,'csv_ghc/summary')